In [103]:
import itertools

numbers = [0,1,2,3]

def power_set(iterable):
    s = list(iterable)
    # Generate combinations for all possible lengths (i in range(len(s) + 1))
    return list(itertools.chain.from_iterable(itertools.combinations(s, i) for i in range(1,len(s) + 1)))

all_subsets = power_set(numbers)
print(all_subsets)

[(0,), (1,), (2,), (3,), (0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3), (0, 1, 2), (0, 1, 3), (0, 2, 3), (1, 2, 3), (0, 1, 2, 3)]


In [104]:
import pandas as pd
import ast
import numpy as np

In [105]:
def softmax(x):
    return np.exp(x) / np.exp(x).sum()

def softmax_dim(x,axis):
    return np.exp(x) / np.sum(np.exp(x),axis=axis,keepdims=True)

In [106]:
df = pd.read_csv("test_multimodel_output.txt", sep="|")

In [107]:
print(df.columns)
#['_'.join(i.split('_')[1:]) for i in list(df.columns) if 'logits_' in i]
logits_columns = [i for i in list(df.columns) if 'logits_' in i]

Index(['image_name', 'pred_label', 'deepfake_score', 'score_convnext',
       'score_clip_cip', 'score_clip_df40pre', 'score_clip_eff_vh',
       'score_clip_eff_vh_2', 'logits_convnext', 'logits_clip_cip',
       'logits_clip_df40pre', 'logits_clip_eff_vh', 'logits_clip_eff_vh_2'],
      dtype='object')


In [108]:
dict_score = {}
for subs_now in all_subsets:
    logs_now = [logits_columns[i] for i in subs_now]
    models_now = ['_'.join(i.split('_')[1:]) for i in logs_now]
    scores_all = []
    for k in logs_now:
        scores_all.append(df[k].apply(eval))
    score_now = [softmax(i)[1] for i in np.sum(scores_all,axis=0)]
    dict_score['|'.join(models_now)]=score_now

In [109]:
dict_df = {}
dict_df["image_name"] = df["image_name"]
dict_df.update(dict_score)


In [110]:
df_result = pd.DataFrame(dict_df)

In [111]:
def flagging_set(x):
    if '23Deepfake' in x: return 'DEEPFAKE'
    elif 'Danamon_20251119' in x: return 'REAL'
    elif 'Ganesha_20250916' in x:  return 'DEEPFAKE'
    elif 'Mobee_20250813' in x:  return 'REAL'
    elif 'Panin_20250925' in x:  return 'REAL'
    elif 'Panin_20250825'  in x:  return 'REAL'
    elif 'Danamon_20250718' in x:  return 'UNKNOWN'
    elif 'Experd_20250908' in x:  return 'REAL'
    elif 'Danamon_20251107' in x:  return 'REAL'
    elif 'Danamon_20251006' in x:  return 'REAL'
    elif 'Experd_20250811' in x:  return 'REAL'
    else:  return 'UNKNOWN'

In [112]:
df_result["label_str"] =df_result["image_name"].apply(lambda x:flagging_set(x.split("/")[10]))
df_result["subgroup"]= df_result["image_name"].apply(lambda x:x.split("/")[10])
df_result["name"]= df_result["image_name"].apply(lambda x:x.split("/")[-1])

In [113]:
dict_label={
    "DEEPFAKE":1,
    "REAL":0,
    "UNKNOWN":-99
}
df_result["label"] = df_result["label_str"].apply(lambda x:dict_label.get(x))

In [114]:
y_pred = df_result[list(dict_score.keys())[0]]

In [115]:
df.head(5)

,image_name,pred_label,deepfake_score,score_convnext,score_clip_cip,score_clip_df40pre,score_clip_eff_vh,score_clip_eff_vh_2,logits_convnext,logits_clip_cip,logits_clip_df40pre,logits_clip_eff_vh,logits_clip_eff_vh_2
0,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,DEEPFAKE,0.91252,0.56828,0.23206,0.99402,0.99948,0.49493,"-0.14901,0.12583","0.82335,-0.37334","-1.62002,3.49335","-2.97154,4.58133","0.53222,0.51192"
1,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,REAL,0.10024,0.99979,0.00115,0.02293,0.05679,0.00221,"-4.21236,4.25979","3.56625,-3.20217","2.79564,-0.95638","2.08288,-0.72714","3.48902,-2.62526"
2,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,DEEPFAKE,0.97409,0.94301,0.02904,0.87394,0.99992,0.99940,"-1.41333,1.39287","2.25576,-1.2537","-0.37389,1.56236","-4.38218,5.10336","-3.06723,4.34795"
3,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,REAL,0.33783,0.99996,0.00028,0.78816,0.78414,0.00033,"-5.48512,4.71404","4.17736,-3.98625","0.13408,1.44793","0.06552,1.35545","4.57675,-3.42752"
4,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,REAL,0.16204,0.99993,0.10592,0.14244,0.01822,0.00005,"-4.90951,4.65184","1.21003,-0.92313","1.83371,0.03856","2.86312,-1.12384","5.05208,-4.80947"


In [116]:
df_result.groupby("subgroup").count()["image_name"]

subgroup
23Deepfake_20250825    23
Danamon_20250718       11
Danamon_20251006       14
Danamon_20251107        1
Danamon_20251119       37
Experd_20250811        92
Experd_20250908        57
Ganesha_20250916        2
Mobee_20250813          5
Panin_20250825          1
Name: image_name, dtype: int64

In [117]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix
)

In [118]:
def compute_eer(y_true, y_score):
    """
    Compute Equal Error Rate (EER)
    """
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    fnr = 1 - tpr
    eer_idx = np.nanargmin(np.abs(fpr - fnr))
    eer = (fpr[eer_idx] + fnr[eer_idx]) / 2
    return eer,thresholds[eer_idx]


def evaluate_binary_classifier(y_true, y_pred, y_score):
    """
    y_true  : ground-truth labels (0 or 1)
    y_pred  : predicted labels (0 or 1)
    y_score : predicted probabilities or confidence scores for class 1
    """

    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    # Metrics
    accuracy  = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall    = recall_score(y_true, y_pred)
    f1        = f1_score(y_true, y_pred)

    avg_prec  = average_precision_score(y_true, y_score)
    roc_auc   = roc_auc_score(y_true, y_score)

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    eer,eer_thresh = compute_eer(y_true, y_score)

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
        "Average Precision": avg_prec,
        "FPR": fpr,
        "FNR": fnr,
        "ROC AUC": roc_auc,
        "EER": eer,
        "EER Threshold": eer_thresh
    }

In [119]:
import itertools

def generate_prob_vectors(classes, step=0.1):
    """
    Generate all probability vectors for given classes
    where values sum to 1.0 with a given step size.
    
    Args:
        classes (list): e.g. [0, 1, 2]
        step (float): resolution (default 0.1)
    
    Returns:
        list[list[float]]
    """
    n = len(classes)
    steps = int(1 / step)

    prob_vectors = []
    for comb in itertools.product(range(steps + 1), repeat=n):
        if sum(comb) == steps:
            prob_vectors.append([round(c * step, 1) for c in comb])

    return prob_vectors

In [120]:
prob_vectors = np.array(generate_prob_vectors([0,1,2,3,4],0.1))

In [121]:
df.head(5)

,image_name,pred_label,deepfake_score,score_convnext,score_clip_cip,score_clip_df40pre,score_clip_eff_vh,score_clip_eff_vh_2,logits_convnext,logits_clip_cip,logits_clip_df40pre,logits_clip_eff_vh,logits_clip_eff_vh_2
0,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,DEEPFAKE,0.91252,0.56828,0.23206,0.99402,0.99948,0.49493,"-0.14901,0.12583","0.82335,-0.37334","-1.62002,3.49335","-2.97154,4.58133","0.53222,0.51192"
1,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,REAL,0.10024,0.99979,0.00115,0.02293,0.05679,0.00221,"-4.21236,4.25979","3.56625,-3.20217","2.79564,-0.95638","2.08288,-0.72714","3.48902,-2.62526"
2,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,DEEPFAKE,0.97409,0.94301,0.02904,0.87394,0.99992,0.99940,"-1.41333,1.39287","2.25576,-1.2537","-0.37389,1.56236","-4.38218,5.10336","-3.06723,4.34795"
3,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,REAL,0.33783,0.99996,0.00028,0.78816,0.78414,0.00033,"-5.48512,4.71404","4.17736,-3.98625","0.13408,1.44793","0.06552,1.35545","4.57675,-3.42752"
4,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,REAL,0.16204,0.99993,0.10592,0.14244,0.01822,0.00005,"-4.90951,4.65184","1.21003,-0.92313","1.83371,0.03856","2.86312,-1.12384","5.05208,-4.80947"


In [122]:
df_res = df.copy()
df_res["label_str"] =df_res["image_name"].apply(lambda x:flagging_set(x.split("/")[10]))
df_res["subgroup"]= df_res["image_name"].apply(lambda x:x.split("/")[10])
df_res["name"]= df_res["image_name"].apply(lambda x:x.split("/")[-1])
dict_label={
    "DEEPFAKE":1,
    "REAL":0,
    "UNKNOWN":-99
}
df_res["label"] = df_res["label_str"].apply(lambda x:dict_label.get(x))
df_res = df_res[df_res["label"].isin([0,1])]
df_res = df_res[df_res["image_name"].apply(lambda x:"experd" not in x.lower())]

In [123]:
df_res.head(5)

,image_name,pred_label,deepfake_score,score_convnext,score_clip_cip,score_clip_df40pre,score_clip_eff_vh,score_clip_eff_vh_2,logits_convnext,logits_clip_cip,logits_clip_df40pre,logits_clip_eff_vh,logits_clip_eff_vh_2,label_str,subgroup,name,label
0,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,DEEPFAKE,0.91252,0.56828,0.23206,0.99402,0.99948,0.49493,"-0.14901,0.12583","0.82335,-0.37334","-1.62002,3.49335","-2.97154,4.58133","0.53222,0.51192",REAL,Danamon_20251119,f955d620-6beb-48ea-bde1-1674cdce22b3.jpg,0
1,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,REAL,0.10024,0.99979,0.00115,0.02293,0.05679,0.00221,"-4.21236,4.25979","3.56625,-3.20217","2.79564,-0.95638","2.08288,-0.72714","3.48902,-2.62526",REAL,Danamon_20251119,e037f79f-abb7-4116-a79a-cf24deb44bd8.jpg,0
2,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,DEEPFAKE,0.97409,0.94301,0.02904,0.87394,0.99992,0.99940,"-1.41333,1.39287","2.25576,-1.2537","-0.37389,1.56236","-4.38218,5.10336","-3.06723,4.34795",REAL,Danamon_20251119,11a11df8-ab47-46b6-bb5b-3b37d5bc7a4c.jpg,0
3,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,REAL,0.33783,0.99996,0.00028,0.78816,0.78414,0.00033,"-5.48512,4.71404","4.17736,-3.98625","0.13408,1.44793","0.06552,1.35545","4.57675,-3.42752",REAL,Danamon_20251119,0e5963f6-f452-42f7-ae52-639585f8092a.jpg,0
4,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,REAL,0.16204,0.99993,0.10592,0.14244,0.01822,0.00005,"-4.90951,4.65184","1.21003,-0.92313","1.83371,0.03856","2.86312,-1.12384","5.05208,-4.80947",REAL,Danamon_20251119,0040178d-1365-43ab-9891-c24415950fb1.jpg,0


In [125]:
thresh_all = [0.5,0.2,0.2,0.2,0.2]
score_cols = [i for i in df_res.columns if i.startswith("score_")]
print(score_cols)
for id_now,col_now in enumerate(score_cols):
    y_true =df_res["label"]
    y_pred = df_res[col_now]>=thresh_all[id_now]
    y_score =df_res[col_now]
    print(col_now + ": "+ str(round(evaluate_binary_classifier(y_true,y_pred,y_score)['ROC AUC'],4)))

['score_convnext', 'score_clip_cip', 'score_clip_df40pre', 'score_clip_eff_vh', 'score_clip_eff_vh_2']
score_convnext: 0.611
score_clip_cip: 0.7476
score_clip_df40pre: 0.6786
score_clip_eff_vh: 0.8469
score_clip_eff_vh_2: 0.9138


In [126]:
evaluate_binary_classifier(y_true,y_pred,y_score)

{'Accuracy': 0.8313253012048193,
 'Precision': 0.8666666666666667,
 'Recall': 0.52,
 'F1-Score': 0.65,
 'Average Precision': 0.7545455628297026,
 'FPR': 0.034482758620689655,
 'FNR': 0.48,
 'ROC AUC': 0.9137931034482758,
 'EER': 0.15758620689655173,
 'EER Threshold': 0.01745}

In [127]:
df_res.groupby("subgroup").count()

,image_name,pred_label,deepfake_score,score_convnext,score_clip_cip,score_clip_df40pre,score_clip_eff_vh,score_clip_eff_vh_2,logits_convnext,logits_clip_cip,logits_clip_df40pre,logits_clip_eff_vh,logits_clip_eff_vh_2,label_str,name,label
subgroup,,,,,,,,,,,,,,,,
23Deepfake_20250825,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23
Danamon_20251006,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14
Danamon_20251107,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
Danamon_20251119,37,37,37,37,37,37,37,37,37,37,37,37,37,37,37,37
Ganesha_20250916,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
Mobee_20250813,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5
Panin_20250825,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1


### Combine with json output

In [128]:
df_res2 = pd.read_csv("test_multimodel_json_combined_deepfake_data_rev_output.txt",sep='|')
df_res2["subgroup"]= df_res2["image_name"].apply(lambda x:x.split("/")[7])

In [129]:
df_res2.head(10)

,image_name,label,pred_label,deepfake_score,time_process_ms,score_convnext,score_clip_cip,score_clip_df40pre,score_clip_eff_vh,score_clip_eff_vh_2,...,logits_clip_cip,logits_clip_df40pre,logits_clip_eff_vh,logits_clip_eff_vh_2,time_ms_convnext,time_ms_clip_cip,time_ms_clip_df40pre,time_ms_clip_eff_vh,time_ms_clip_eff_vh_2,subgroup
0,/mnt/SSD/dataset/deepfake/data_client/processe...,1,DEEPFAKE,0.98904,92.60,0.99252,0.99978,0.99614,0.99820,0.06493,...,"-3.96526,4.45179","-2.26216,3.29176","-2.48515,3.83383","1.57981,-1.08746",31.01,6.09,13.93,13.34,13.84,data_deepfake_bcad
1,/mnt/SSD/dataset/deepfake/data_client/processe...,1,DEEPFAKE,0.99680,72.07,0.99449,0.99941,0.99983,0.99805,0.76211,...,"-3.31839,4.12494","-3.982,4.68549","-2.52407,3.71461","-0.25814,0.90612",21.67,4.12,11.32,11.36,11.40,data_deepfake_bcad
2,/mnt/SSD/dataset/deepfake/data_client/processe...,1,DEEPFAKE,0.99024,73.34,0.61663,0.99680,0.99993,0.99627,0.84125,...,"-2.70345,3.03698","-4.41665,5.21159","-2.19251,3.39545","-0.63349,1.03405",21.80,4.04,11.79,11.84,11.69,data_deepfake_bcad
3,/mnt/SSD/dataset/deepfake/data_client/processe...,1,REAL,0.17656,73.05,0.99688,0.08330,0.99161,0.00171,0.00008,...,"1.09674,-1.30161","-1.61536,3.1575","3.82004,-2.55009","4.87337,-4.59626",21.88,4.12,11.42,11.53,11.57,data_deepfake_bcad
4,/mnt/SSD/dataset/deepfake/data_client/processe...,1,DEEPFAKE,0.97206,72.16,1.00000,0.99432,0.99085,0.13501,0.05243,...,"-2.25943,2.90605","-1.64401,3.04072","1.6212,-0.23613","1.65591,-1.23855",21.67,4.10,11.38,11.45,11.42,data_deepfake_bcad
5,/mnt/SSD/dataset/deepfake/data_client/processe...,1,DEEPFAKE,0.98364,73.98,0.99501,0.37178,0.99980,0.99676,0.80844,...,"0.32686,-0.19772","-3.51906,5.02319","-1.96265,3.76768","-0.34593,1.09399",21.72,4.23,11.77,11.85,11.76,data_deepfake_bcad
6,/mnt/SSD/dataset/deepfake/data_client/processe...,1,DEEPFAKE,0.99959,73.15,0.99999,0.99883,0.99972,0.99881,0.99670,...,"-3.06989,3.68337","-3.48227,4.68789","-2.68752,4.04191","-2.6111,3.10022",21.99,4.12,11.47,11.60,11.55,data_deepfake_bcad
7,/mnt/SSD/dataset/deepfake/data_client/processe...,1,DEEPFAKE,0.80564,72.02,0.99907,0.01544,0.99886,0.79921,0.02033,...,"2.48078,-1.67445","-2.73453,4.0422","0.04802,1.42938","2.12392,-1.75138",21.65,4.06,11.30,11.33,11.40,data_deepfake_bcad
8,/mnt/SSD/dataset/deepfake/data_client/processe...,1,DEEPFAKE,0.93395,73.83,0.99626,0.91423,0.99807,0.95582,0.01751,...,"-1.11892,1.24753","-2.56252,3.68534","-0.87483,2.19943","2.18328,-1.84425",21.94,4.18,11.74,11.82,11.74,data_deepfake_bcad
9,/mnt/SSD/dataset/deepfake/data_client/processe...,1,DEEPFAKE,0.80676,72.99,0.99994,0.26547,0.92225,0.44368,0.02149,...,"0.74522,-0.27253","-0.06346,2.4099","1.12961,0.90335","2.2084,-1.61016",21.88,4.18,11.41,11.52,11.69,data_deepfake_bcad


In [130]:
df_res2["subgroup"].unique()

array(['data_deepfake_bcad', 'data_deepfake_maybank',
       'data_deepfake_raya'], dtype=object)

In [138]:
df_res_raw = df_res.copy()
df_res1 = df_res_raw[["image_name","subgroup","label","logits_convnext","logits_clip_cip","logits_clip_df40pre","logits_clip_eff_vh","logits_clip_eff_vh_2"]].copy()
df_res = pd.concat([df_res1[["image_name","subgroup","label","logits_convnext","logits_clip_cip","logits_clip_df40pre","logits_clip_eff_vh","logits_clip_eff_vh_2"]],
df_res2[["image_name","subgroup","label","logits_convnext","logits_clip_cip","logits_clip_df40pre","logits_clip_eff_vh","logits_clip_eff_vh_2"]]])

In [139]:
df_res1 = df_res_raw[["image_name","subgroup","label","logits_convnext","logits_clip_cip","logits_clip_df40pre","logits_clip_eff_vh","logits_clip_eff_vh_2"
                  ,"score_convnext","score_clip_cip","score_clip_df40pre","score_clip_eff_vh","score_clip_eff_vh_2"]].copy()
df_res_all = pd.concat([df_res1[["image_name","subgroup","label","logits_convnext","logits_clip_cip","logits_clip_df40pre","logits_clip_eff_vh","logits_clip_eff_vh_2"
                                 ,"score_convnext","score_clip_cip","score_clip_df40pre","score_clip_eff_vh","score_clip_eff_vh_2"]],
df_res2[["image_name","subgroup","label","logits_convnext","logits_clip_cip","logits_clip_df40pre","logits_clip_eff_vh","logits_clip_eff_vh_2"
         ,"score_convnext","score_clip_cip","score_clip_df40pre","score_clip_eff_vh","score_clip_eff_vh_2"]]])

In [140]:
df_res.head(5)

,image_name,subgroup,label,logits_convnext,logits_clip_cip,logits_clip_df40pre,logits_clip_eff_vh,logits_clip_eff_vh_2
0,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,Danamon_20251119,0,"-0.14901,0.12583","0.82335,-0.37334","-1.62002,3.49335","-2.97154,4.58133","0.53222,0.51192"
1,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,Danamon_20251119,0,"-4.21236,4.25979","3.56625,-3.20217","2.79564,-0.95638","2.08288,-0.72714","3.48902,-2.62526"
2,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,Danamon_20251119,0,"-1.41333,1.39287","2.25576,-1.2537","-0.37389,1.56236","-4.38218,5.10336","-3.06723,4.34795"
3,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,Danamon_20251119,0,"-5.48512,4.71404","4.17736,-3.98625","0.13408,1.44793","0.06552,1.35545","4.57675,-3.42752"
4,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,Danamon_20251119,0,"-4.90951,4.65184","1.21003,-0.92313","1.83371,0.03856","2.86312,-1.12384","5.05208,-4.80947"


In [141]:
df_res.groupby(['subgroup','label']).count()["image_name"]

subgroup               label
23Deepfake_20250825    1         23
Danamon_20251006       0         14
Danamon_20251107       0          1
Danamon_20251119       0         37
Ganesha_20250916       1          2
Mobee_20250813         0          5
Panin_20250825         0          1
data_deepfake_bcad     1         29
data_deepfake_maybank  0        111
data_deepfake_raya     1         10
Name: image_name, dtype: int64

In [142]:
thresh_all = [0.5,0.2,0.2,0.2,0.2]
score_cols = [i for i in df_res_all.columns if i.startswith("score_")]
print(score_cols)
for id_now,col_now in enumerate(score_cols):
    y_true =df_res_all["label"]
    y_pred = df_res_all[col_now]>=thresh_all[id_now]
    y_score =df_res_all[col_now]
    print(col_now + ": "+ str(round(evaluate_binary_classifier(y_true,y_pred,y_score)['ROC AUC'],4)))

['score_convnext', 'score_clip_cip', 'score_clip_df40pre', 'score_clip_eff_vh', 'score_clip_eff_vh_2']
score_convnext: 0.785
score_clip_cip: 0.9497
score_clip_df40pre: 0.8317
score_clip_eff_vh: 0.8957
score_clip_eff_vh_2: 0.8954


In [143]:
df_res_all.head()

,image_name,subgroup,label,logits_convnext,logits_clip_cip,logits_clip_df40pre,logits_clip_eff_vh,logits_clip_eff_vh_2,score_convnext,score_clip_cip,score_clip_df40pre,score_clip_eff_vh,score_clip_eff_vh_2
0,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,Danamon_20251119,0,"-0.14901,0.12583","0.82335,-0.37334","-1.62002,3.49335","-2.97154,4.58133","0.53222,0.51192",0.56828,0.23206,0.99402,0.99948,0.49493
1,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,Danamon_20251119,0,"-4.21236,4.25979","3.56625,-3.20217","2.79564,-0.95638","2.08288,-0.72714","3.48902,-2.62526",0.99979,0.00115,0.02293,0.05679,0.00221
2,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,Danamon_20251119,0,"-1.41333,1.39287","2.25576,-1.2537","-0.37389,1.56236","-4.38218,5.10336","-3.06723,4.34795",0.94301,0.02904,0.87394,0.99992,0.99940
3,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,Danamon_20251119,0,"-5.48512,4.71404","4.17736,-3.98625","0.13408,1.44793","0.06552,1.35545","4.57675,-3.42752",0.99996,0.00028,0.78816,0.78414,0.00033
4,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,Danamon_20251119,0,"-4.90951,4.65184","1.21003,-0.92313","1.83371,0.03856","2.86312,-1.12384","5.05208,-4.80947",0.99993,0.10592,0.14244,0.01822,0.00005


### Calculate Metrics

In [144]:
mat_feat = np.array([np.array([np.array(ast.literal_eval(i)) for i in list(df_res["logits_convnext"])]),
np.array([np.array(ast.literal_eval(i)) for i in list(df_res["logits_clip_cip"])]),
np.array([np.array(ast.literal_eval(i)) for i in list(df_res["logits_clip_df40pre"])]),
np.array([np.array(ast.literal_eval(i)) for i in list(df_res["logits_clip_eff_vh"])]),
np.array([np.array(ast.literal_eval(i)) for i in list(df_res["logits_clip_eff_vh_2"])]),
])
#np.array([np.array(ast.literal_eval(i)) for i in list(df_res["logits_convnext"])])
#np.array([np.array(ast.literal_eval(i)) for i in list(df_res["logits_convnext"])])
#np.array([np.array(ast.literal_eval(i)) for i in list(df_res["logits_convnext"])])

In [145]:
print(prob_vectors.shape)
print(mat_feat.shape)

(1001, 5)
(5, 233, 2)


In [146]:
# ConvNext Only
res_0 = np.matmul([[1.0,0,0,0,0]],mat_feat[:,:,0])
res_1 = np.matmul([[1.0,0,0,0,0]],mat_feat[:,:,1])
roc_all = []
f1_all =[]
metrics_all = []
for i in range(len(res_0)): # Loop over all combinations
    logit = np.stack([res_0[i],res_1[i]],1)
    y_score = np.array([softmax_dim(i,-1)[1] for i in logit])
    y_pred = y_score>0.75
    y_true = df_res["label"]
    metrics = evaluate_binary_classifier(y_true, y_pred, y_score)
    roc_all.append(metrics["ROC AUC"])
    f1_all.append(metrics["F1-Score"])
    metrics_all.append(metrics)
    #print(f"{mod_now}: {metrics}")

print(np.argwhere(f1_all == np.amax(f1_all)))
print(metrics_all[0])

[[0]]
{'Accuracy': 0.6909871244635193, 'Precision': 0.46226415094339623, 'Recall': 0.765625, 'F1-Score': 0.5764705882352942, 'Average Precision': 0.5558837563248763, 'FPR': 0.33727810650887574, 'FNR': 0.234375, 'ROC AUC': 0.7849482248520709, 'EER': 0.27671967455621305, 'EER Threshold': 0.9481766497710454}


In [148]:
df_res_all['score_convnext']

0      0.56828
1      0.99979
2      0.94301
3      0.99996
4      0.99993
        ...   
145    0.94818
146    0.61970
147    0.99997
148    0.99591
149    0.99915
Name: score_convnext, Length: 233, dtype: float64

In [149]:
y_score

0      0.56828
1      0.99979
2      0.94301
3      0.99996
4      0.99993
        ...   
145    0.94818
146    0.61970
147    0.99997
148    0.99591
149    0.99915
Name: score_convnext, Length: 233, dtype: float64

DFD 1.0.0 Performance

In [150]:
res_0 = np.matmul([[0.5,0.5,0,0,0]],mat_feat[:,:,0])
res_1 = np.matmul([[0.5,0.5,0,0,0]],mat_feat[:,:,1])
roc_all = []
f1_all =[]
metrics_all = []
for i in range(len(res_0)): # Loop over all combinations
    logit = np.stack([res_0[i],res_1[i]],1)
    y_score = np.array([softmax_dim(i,-1)[1] for i in logit])
    y_pred = y_score>0.75
    y_true = df_res["label"]
    metrics = evaluate_binary_classifier(y_true, y_pred, y_score)
    roc_all.append(metrics["ROC AUC"])
    f1_all.append(metrics["F1-Score"])
    metrics_all.append(metrics)
    #print(f"{mod_now}: {metrics}")

print(np.argwhere(f1_all == np.amax(f1_all)))
print(metrics_all[0])

[[0]]
{'Accuracy': 0.8669527896995708, 'Precision': 0.7538461538461538, 'Recall': 0.765625, 'F1-Score': 0.7596899224806202, 'Average Precision': 0.8354497332618092, 'FPR': 0.09467455621301775, 'FNR': 0.234375, 'ROC AUC': 0.9225221893491125, 'EER': 0.18652921597633138, 'EER Threshold': 0.5113405547487052}


In [152]:
prob_vectors_2=generate_prob_vectors([0,1], step=0.1)
res_0 = np.matmul(prob_vectors_2,mat_feat[[0,1]][:,:,0])
res_1 = np.matmul(prob_vectors_2,mat_feat[[0,1]][:,:,1])
roc_all = []
f1_all =[]
metrics_all = []
for i in range(len(res_0)): # Loop over all combinations
    logit = np.stack([res_0[i],res_1[i]],1)
    y_score = np.array([softmax(i)[1] for i in logit])
    y_pred = y_score>0.5
    y_true = df_res["label"]
    metrics = evaluate_binary_classifier(y_true, y_pred, y_score)
    roc_all.append(metrics["ROC AUC"])
    f1_all.append(metrics["F1-Score"])
    metrics_all.append(metrics)
    #print(f"{mod_now}: {metrics}")

print(np.argwhere(f1_all == np.amax(f1_all)))
print(prob_vectors_2[4])
print(metrics_all[4])

[[4]]
[0.4, 0.6]
{'Accuracy': 0.8755364806866953, 'Precision': 0.7692307692307693, 'Recall': 0.78125, 'F1-Score': 0.7751937984496126, 'Average Precision': 0.8661056535917452, 'FPR': 0.08875739644970414, 'FNR': 0.21875, 'ROC AUC': 0.9367603550295858, 'EER': 0.14131841715976332, 'EER Threshold': 0.3260546542157721}


DFD 1.2.0 Alpha performance

In [153]:
res_0 = np.matmul(prob_vectors,mat_feat[:,:,0])
res_1 = np.matmul(prob_vectors,mat_feat[:,:,1])
roc_all = []
f1_all =[]
metrics_all = []
for thresh_now in np.arange(0.3,0.7,0.1):
    for i in range(len(res_0)): # Loop over all combinations
        logit = np.stack([res_0[i],res_1[i]],1)
        y_score = np.array([softmax(i)[1] for i in logit])
        y_pred = y_score>thresh_now
        y_true = df_res["label"]
        metrics = evaluate_binary_classifier(y_true, y_pred, y_score)
        roc_all.append(metrics["ROC AUC"])
        f1_all.append(metrics["F1-Score"])
        metrics_all.append(metrics)
        #print(f"{mod_now}: {metrics}")

id_max = np.argwhere(f1_all == np.amax(f1_all))[0][0]
print(metrics_all[id_max])
prob_vectors[id_max%len(res_0)]

{'Accuracy': 0.9098712446351931, 'Precision': 0.8307692307692308, 'Recall': 0.84375, 'F1-Score': 0.8372093023255814, 'Average Precision': 0.9112998176592857, 'FPR': 0.0650887573964497, 'FNR': 0.15625, 'ROC AUC': 0.9574704142011834, 'EER': 0.1109005177514793, 'EER Threshold': 0.19425587437467723}


array([0.2, 0.5, 0.1, 0.2, 0. ])

In [154]:
print(id_max)
print(id_max%len(res_0))
print(int(id_max/len(res_0)))

657
657
0


In [155]:
print(np.arange(0,0.5,0.1)[int(id_max/(len(res_0)))])

0.0


In [156]:
y_score.shape

(233,)

DFD 1.2.0 - No Clip Ciplab

In [157]:
mat_feat[[0,2,3]].shape

(3, 233, 2)

In [158]:
prob_vectors_3=generate_prob_vectors([0,1,2], step=0.1)
res_0 = np.matmul(prob_vectors_3,mat_feat[[0,2,3]][:,:,0])
res_1 = np.matmul(prob_vectors_3,mat_feat[[0,2,3]][:,:,1])
roc_all = []
f1_all =[]
metrics_all = []
for i in range(len(res_0)): # Loop over all combinations
    logit = np.stack([res_0[i],res_1[i]],1)
    y_score = np.array([softmax(i)[1] for i in logit])
    y_pred = y_score>0.5
    y_true = df_res["label"]
    metrics = evaluate_binary_classifier(y_true, y_pred, y_score)
    roc_all.append(metrics["ROC AUC"])
    f1_all.append(metrics["F1-Score"])
    metrics_all.append(metrics)
    #print(f"{mod_now}: {metrics}")

print(np.argwhere(f1_all == np.amax(f1_all)))
print(metrics_all[24])
prob_vectors_3[24]

[[13]]
{'Accuracy': 0.8369098712446352, 'Precision': 0.6511627906976745, 'Recall': 0.875, 'F1-Score': 0.7466666666666667, 'Average Precision': 0.8321788169236098, 'FPR': 0.17751479289940827, 'FNR': 0.125, 'ROC AUC': 0.9209504437869822, 'EER': 0.1442769970414201, 'EER Threshold': 0.6532248915340705}


[0.2, 0.3, 0.5]

DFD 1.2.0 - No Convnext

In [164]:
prob_vectors_3=generate_prob_vectors([0,1,2,3], step=0.1)
res_0 = np.matmul(prob_vectors_3,mat_feat[1:,:,0])
res_1 = np.matmul(prob_vectors_3,mat_feat[1:,:,1])
roc_all = []
f1_all =[]
metrics_all = []
vec_all = []
thresh_all = []
for i in range(len(res_0)): # Loop over all combinations
    vec_now = prob_vectors_3[i]
    logit = np.stack([res_0[i],res_1[i]],1)
    y_score = np.array([softmax(i)[1] for i in logit])
    for thresh_now in np.arange(0.2,0.7,0.1):
        y_pred = y_score>thresh_now
        y_true = df_res["label"]
        metrics = evaluate_binary_classifier(y_true, y_pred, y_score)
        roc_all.append(metrics["ROC AUC"])
        f1_all.append(metrics["F1-Score"])
        metrics_all.append(metrics)
        vec_all.append(vec_now)
        thresh_all.append(thresh_now)
        #print(f"{mod_now}: {metrics}")

print(np.argwhere(f1_all == np.amax(f1_all)))
id_optimum =np.argwhere(f1_all == np.amax(f1_all))[0][0]
print(metrics_all[id_optimum])
print(vec_all[id_optimum])
print(thresh_all[id_optimum])
#prob_vectors_3[id_optimum]
#prob_vectors_3[id_optimum%len(res_0)]
#id_max = np.argwhere(f1_all == np.amax(f1_all))[0][0]
#print(metrics_all[id_max])
#prob_vectors[id_max%len(res_0)]

[[1235]]
{'Accuracy': 0.9012875536480687, 'Precision': 0.8253968253968254, 'Recall': 0.8125, 'F1-Score': 0.8188976377952756, 'Average Precision': 0.8996714326456994, 'FPR': 0.0650887573964497, 'FNR': 0.1875, 'ROC AUC': 0.9514607988165681, 'EER': 0.1246301775147929, 'EER Threshold': 0.0748091387081091}
[0.5, 0.3, 0.2, 0.0]
0.2


DFD 1.2.0 - Only DF40 & Effort

In [160]:
prob_vectors_3=generate_prob_vectors([0,1,2], step=0.1)
res_0 = np.matmul(prob_vectors_3,mat_feat[2:,:,0])
res_1 = np.matmul(prob_vectors_3,mat_feat[2:,:,1])
roc_all = []
f1_all =[]
metrics_all = []
vec_all = []
for i in range(len(res_0)): # Loop over all combinations
    vec_now = prob_vectors_3[i]
    logit = np.stack([res_0[i],res_1[i]],1)
    y_score = np.array([softmax(i)[1] for i in logit])
    for thresh_now in np.arange(0.1,0.7,0.1):
        y_pred = y_score>thresh_now
        y_true = df_res["label"]
        metrics = evaluate_binary_classifier(y_true, y_pred, y_score)
        roc_all.append(metrics["ROC AUC"])
        f1_all.append(metrics["F1-Score"])
        metrics_all.append(metrics)
        vec_all.append(vec_now)
        #print(f"{mod_now}: {metrics}")

print(np.argwhere(f1_all == np.amax(f1_all)))
id_optimum =np.argwhere(f1_all == np.amax(f1_all))[0][0]
print(metrics_all[id_optimum])
print(vec_all[id_optimum])
#prob_vectors_3[id_optimum]
#prob_vectors_3[id_optimum%len(res_0)]
#id_max = np.argwhere(f1_all == np.amax(f1_all))[0][0]
#print(metrics_all[id_max])
#prob_vectors[id_max%len(res_0)]

[[78]]
{'Accuracy': 0.8798283261802575, 'Precision': 0.78125, 'Recall': 0.78125, 'F1-Score': 0.78125, 'Average Precision': 0.7955494902455617, 'FPR': 0.08284023668639054, 'FNR': 0.21875, 'ROC AUC': 0.9126294378698224, 'EER': 0.1275887573964497, 'EER Threshold': 0.028770897851972576}
[0.1, 0.2, 0.7]


In [161]:
int(id_optimum/len(res_0))

1

In [162]:
#res_0 = np.matmul(prob_vectors,mat_feat[:,:,0])
#res_1 = np.matmul(prob_vectors,mat_feat[:,:,1])
res = softmax_dim(mat_feat,axis=-1)[:,:,1]
res = np.matmul(prob_vectors,res)
roc_all = []
f1_all =[]
metrics_all = []
for i in range(len(res_0)):
#    #logit = np.stack([res_0[i],res_1[i]],1)
    y_score = np.array(res[i])
    y_pred = y_score>0.1
    y_true = df_res["label"]
    metrics = evaluate_binary_classifier(y_true, y_pred, y_score)
    roc_all.append(metrics["ROC AUC"])
    f1_all.append(metrics["F1-Score"])
    metrics_all.append(metrics)
    #print(f"{mod_now}: {metrics}")

In [90]:
#prob_vectors
#res[0]

In [91]:
print(np.argmax(roc_all))
print(prob_vectors[0])
print(metrics_all[0])
#print(metrics_all[203])
#prob_vectors[203]

0
[0. 0. 0. 1.]
{'Accuracy': 0.8072289156626506, 'Precision': 0.7647058823529411, 'Recall': 0.52, 'F1-Score': 0.6190476190476191, 'Average Precision': 0.7545455628297026, 'FPR': 0.06896551724137931, 'FNR': 0.48, 'ROC AUC': 0.9137931034482758, 'EER': 0.15758620689655173}


In [69]:
score

NameError: name 'score' is not defined

In [ ]:
np.matmul(prob_vectors,feat_now)

TypeError: can't multiply sequence by non-int of type 'float'

In [ ]:
ab = df_res["logits_convnext"]

In [ ]:
np.matmul(np.expand_dims(np.array(prob_vectors[0]),0),feat_now)

TypeError: only length-1 arrays can be converted to Python scalars

In [50]:
feat_now.dtype

dtype('O')

In [20]:
df_res = df_result[df_result["label"].isin([0,1])]
df_res = df_res[df_res["image_name"].apply(lambda x:"experd" not in x.lower())]
for mod_now in list(dict_score.keys()):
    y_score = df_res[mod_now]
    y_pred = y_score>0.5
    y_true = df_res["label"]
    metrics = evaluate_binary_classifier(y_true, y_pred, y_score)
    print(f"{mod_now}: {metrics}")

convnext: {'Accuracy': 0.39285714285714285, 'Precision': 0.3194444444444444, 'Recall': 0.92, 'F1-Score': 0.47422680412371127, 'Average Precision': 0.4898317138205554, 'FPR': 0.8305084745762712, 'FNR': 0.08, 'ROC AUC': 0.616271186440678, 'EER': 0.4488135593220339}
clip_cip: {'Accuracy': 0.7261904761904762, 'Precision': 1.0, 'Recall': 0.08, 'F1-Score': 0.14814814814814814, 'Average Precision': 0.5449611193740224, 'FPR': 0.0, 'FNR': 0.92, 'ROC AUC': 0.7450847457627119, 'EER': 0.3494915254237288}
clip_df40pre: {'Accuracy': 0.7142857142857143, 'Precision': 0.5555555555555556, 'Recall': 0.2, 'F1-Score': 0.29411764705882354, 'Average Precision': 0.5572412076019752, 'FPR': 0.06779661016949153, 'FNR': 0.8, 'ROC AUC': 0.7071186440677967, 'EER': 0.3125423728813559}
clip_eff_vh: {'Accuracy': 0.7738095238095238, 'Precision': 0.875, 'Recall': 0.28, 'F1-Score': 0.42424242424242425, 'Average Precision': 0.7519851140075926, 'FPR': 0.01694915254237288, 'FNR': 0.72, 'ROC AUC': 0.912542372881356, 'EER': 0

In [18]:
df_res = df_result[df_result["label"].isin([0,1])]
for mod_now in list(dict_score.keys()):
    y_score = df_res[mod_now]
    y_pred = y_score>0.75
    y_true = df_res["label"]
    metrics = evaluate_binary_classifier(y_true, y_pred, y_score)
    print(f"{mod_now}: {metrics}")

convnext: {'Accuracy': 0.2543103448275862, 'Precision': 0.11458333333333333, 'Recall': 0.88, 'F1-Score': 0.20276497695852533, 'Average Precision': 0.3625060352768711, 'FPR': 0.821256038647343, 'FNR': 0.12, 'ROC AUC': 0.6997101449275362, 'EER': 0.3635748792270531}
clip_cip: {'Accuracy': 0.8491379310344828, 'Precision': 0.0, 'Recall': 0.0, 'F1-Score': 0.0, 'Average Precision': 0.10430500215010219, 'FPR': 0.04830917874396135, 'FNR': 1.0, 'ROC AUC': 0.4807729468599034, 'EER': 0.5208695652173914}
clip_df40pre: {'Accuracy': 0.8103448275862069, 'Precision': 0.14814814814814814, 'Recall': 0.16, 'F1-Score': 0.15384615384615383, 'Average Precision': 0.19184531024224669, 'FPR': 0.1111111111111111, 'FNR': 0.84, 'ROC AUC': 0.4859903381642513, 'EER': 0.5184541062801933}
clip_eff_vh: {'Accuracy': 0.8663793103448276, 'Precision': 0.3125, 'Recall': 0.2, 'F1-Score': 0.24390243902439027, 'Average Precision': 0.2419337912005013, 'FPR': 0.05314009661835749, 'FNR': 0.8, 'ROC AUC': 0.7053140096618358, 'EER':

In [ ]:
# Example inputs
y_true  = np.array([0, 1, 1, 0, 1, 0])
y_pred  = np.array([0, 1, 0, 0, 1, 1])
y_score = np.array([0.1, 0.9, 0.4, 0.2, 0.8, 0.6])

metrics = evaluate_binary_classifier(y_true, y_pred, y_score)

In [64]:
#list(set(df_result["image_name"].apply(lambda x:flagging_set(x.split("/")[10]))))
df_result["image_name"].apply(lambda x:flagging_set(x.split("/")[10]))

0         REAL
1         REAL
2         REAL
3         REAL
4         REAL
        ...   
243    UNKNOWN
244    UNKNOWN
245    UNKNOWN
246    UNKNOWN
247    UNKNOWN
Name: image_name, Length: 248, dtype: object

In [44]:
pd.DataFrame([df["image_name"],*dict_score.values()])

AttributeError: 'builtin_function_or_method' object has no attribute 'get_indexer'

In [ ]:

df[["image_name"]+logits_columns]

,image_name,logits_convnext,logits_clip_cip,logits_clip_df40pre,logits_clip_eff_vh
0,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,"-0.14885,0.12561","0.82292,-0.3723","2.82977,-1.3553","0.52818,0.51469"
1,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,"-4.21294,4.26021","3.56715,-3.20384","3.32163,-2.66661","3.49105,-2.62699"
2,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,"-1.4124,1.39197","2.2563,-1.25485","2.89255,-1.68316","-3.0658,4.34631"
3,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,"-5.48576,4.71469","4.17789,-3.98811","4.52538,-2.88931","4.57531,-3.42503"
4,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,"-4.91082,4.65324","1.2102,-0.92306","4.32508,-3.06615","5.05061,-4.80767"
...,...,...,...,...,...
243,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,"5.30018,-5.87437","2.00109,-1.74565","0.45406,1.32089","3.73602,-1.77929"
244,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,"-1.2508,1.11589","2.64981,-2.54093","4.41395,-2.95696","5.01567,-3.99137"
245,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,"-3.60066,3.74731","3.41912,-3.7737","3.57877,-1.71152","4.6593,-3.60352"
246,/mnt/HDD/workspace/adi/repos/vh_deepfake_train...,"-1.66108,1.63977","1.627,-1.58129","2.8222,-1.328","1.34473,-0.19084"
